## negocios llamadas por mes para el funnel

USE THOTH;
GO

DROP TABLE IF EXISTS
    Tmp_Llamadas_64,
    Tmp_LLamadas_Efectiva_Negocios_1,
    Tmp_LLamadas_Efectiva_Negocios_2,
    Tmp_LLamadas_Efectiva_Negocios_3,
    Tmp_LLamadas_Efectiva_Negocios_ref,
    Tmp_LLamadas_Efectiva_Negocios_ref_5,
    #Tmp_LLamadas_Efectiva_Negocios_5,
    #RHC,
    Tmp_LLamadas_Efectiva_Negocios_4,
    Tmp_LLamadas_Efectiva_Negocios;

DECLARE @HOY DATE = GETDATE();
DECLARE @inicio DATE = DATEADD(DAY, 1, EOMONTH(@HOY, -3));

DECLARE @CAMPANA NVARCHAR(50) =
    CONCAT(
        YEAR(@inicio),
        '-',
        RIGHT('0' + CAST(MONTH(@inicio) AS VARCHAR(2)), 2),
        ' EFECTIVA_NEGOCIOS'
    );
/*=======================================================================================*/  
/*=======================================================================================*/  
/*=======================================================================================*/  
Select a.*,  
b.TIPO as Estado_,  
b.SUB_DESCRIPCION as Sub_Estado_,  
b.DESCRIPCION as Descripcion_,  
b.[PESO] as Pesos,  
right('00000000'+a.dni,8)+' '+a.PHONE_NUMBER AS Enlace,  
a.Fecha_Llamada as Fecha_Llam,  
DATEPART(hh,a.Fecha_Hora_Llamada) as Hora_Llamada,  
[RH]=case when Trama_Hora >= 15 then 3 when Trama_Hora >= 12 then 2 else 1 END,  
b.CODIGO_BCO as COD_BCO  
Into Tmp_LLamadas_Efectiva_Negocios_ref  
From SAMANTHA..tmp_llamadas_mes a WITH (NOLOCK)   
LEFT JOIN ODIN.[dbo].[tTipologia_Efectiva] b WITH (NOLOCK) on a.Codigo_Paleta=b.CODIGO  
where a.Fecha_Llamada >=@inicio
and a.Fecha_Llamada < EOMONTH(@inicio)
and a.Nombre_Campana IN(@CAMPANA);  
/*=======================================================================================*/  
/*=======================================================================================*/  
Select row_number() OVER (PARTITION BY Enlace ORDER BY convert(int,Pesos) asc) AS Mejor_Telefono , * INTO Tmp_LLamadas_Efectiva_Negocios_1 FROM Tmp_LLamadas_Efectiva_Negocios_ref WITH (NOLOCK);   
Select row_number() OVER (PARTITION BY dni ORDER BY convert(int,Pesos) asc,Fecha_Llamada Desc) AS Mejor_Resultado , * INTO Tmp_LLamadas_Efectiva_Negocios_2 FROM Tmp_LLamadas_Efectiva_Negocios_1 WITH (NOLOCK);  
/*=======================================================================================*/  
Select * Into #Tmp_LLamadas_Efectiva_Negocios_5 from Tmp_LLamadas_Efectiva_Negocios_2 WITH (NOLOCK) where Mejor_Resultado=1;  
  
select DISTINCT DNI,RH INTO #RHC from Tmp_LLamadas_Efectiva_Negocios_ref ORDER BY DNI  
  
SELECT D.*,CONCAT(ISNULL(B.RH,''),ISNULL(C.RH,''),ISNULL(E.RH,''))[RHFC]  
INTO Tmp_LLamadas_Efectiva_Negocios_ref_5   
FROM #Tmp_LLamadas_Efectiva_Negocios_5 D WITH (NOLOCK)  
LEFT JOIN (SELECT * FROM #RHC WHERE RH = 1) B ON D.Dni=B.Dni  
LEFT JOIN (SELECT * FROM #RHC WHERE RH = 2) C ON D.Dni=C.Dni  
LEFT JOIN (SELECT * FROM #RHC WHERE RH = 3) E ON D.Dni=E.Dni  
  
CREATE INDEX DNI ON  Tmp_LLamadas_Efectiva_Negocios_ref_5 (DNI);  
/*=======================================================================================*/  
/*=======================================================================================*/  
DROP TABLE IF EXISTS
    dbo.Tmp_Llamadas_64,
    dbo.Tmp_LLamadas_Efectiva_Negocios_1,
    dbo.Tmp_LLamadas_Efectiva_Negocios_2,
    dbo.Tmp_LLamadas_Efectiva_Negocios_3,
    dbo.Tmp_LLamadas_Efectiva_Negocios_4;